# nmtc-application-builder — Full Application Walkthrough

**Week 2 of 4: Section Generators, Table Builders & Output Renderers**

This notebook demonstrates the complete Week 2 workflow.
A CDE can produce a competition-ready application package with a single `generate()` call.

1. Build a CDE profile and pipeline
2. Run `app.analyze()` for intelligence
3. Inspect section generator output (Sections A-E)
4. Inspect table builder output (6 supporting tables)
5. Call `app.generate()` for Word, Excel, PDF, and Markdown
6. Verify the generated files

>  All calls work offline. External APIs fall back to sample data automatically.

In [1]:
import sys, os
sys.path.insert(0, '..')

from nmtcapp.core.application import Application
from nmtcapp.core.cde import CDEProfile
from nmtcapp.core.pipeline import Pipeline
import pandas as pd

print("nmtc-application-builder imports OK")

nmtc-application-builder imports OK


## 1. Build CDE Profile and Pipeline

In [2]:
cde = CDEProfile.sample()
pipeline = Pipeline.sample(n=20)

app = Application(
    cde=cde,
    requested_allocation=65_000_000,
    application_round="CY2025",
)
app.add_pipeline(pipeline)

print(f"CDE:        {cde.name}")
print(f"Pipeline:   {len(pipeline)} projects")
print(f"Total QEI:  ${sum(p.qei_request for p in pipeline):,.0f}")

CDE:        Heartland Impact CDE, LLC
Pipeline:   20 projects
Total QEI:  $122,500,000


## 2. Run Analysis

In [3]:
analysis = app.analyze()
score = analysis.readiness_score

print(f"Readiness Score:      {score.overall_score:.1f}/100 (Grade {score.grade})")
print(f"Deep/Severe Distress: {analysis.distress_analysis['pct_deep_or_severe']:.0%}")
print(f"States:               {analysis.geographic_analysis['states_count']}")
print(f"Jobs Created:         {analysis.impact_summary['total_jobs_created']:,}")

Readiness Score:      86.6/100 (Grade A)
Deep/Severe Distress: 87%
States:               20
Jobs Created:         864


## 3. Section Generator Output

In [4]:
from nmtcapp.sections import ALL_SECTIONS

for section_gen in ALL_SECTIONS:
    content = section_gen.generate_content(app, analysis)
    n_subs = len(content["subsections"])
    print(f"Section {content['section_id']}: {content['title']}  ({n_subs} subsections)")

Section A: Business Strategy  (5 subsections)
Section B: Community Outcomes  (5 subsections)
Section C: Management Capacity  (5 subsections)
Section D: Capitalization Strategy  (5 subsections)
Section E: Prior Awards — Deployment History  (4 subsections)


### Section A — Investment Thesis Narrative

In [5]:
from nmtcapp.sections.section_a_business import SectionABusinessStrategy

content_a = SectionABusinessStrategy().generate_content(app, analysis)
thesis = next(s for s in content_a["subsections"] if "thesis" in s["heading"].lower())
print(f"### {thesis['heading']}\n")
print(thesis["body"][:600], "...")

### Investment Thesis and Strategy

Heartland Impact CDE, LLC will deploy $65.0 million in NMTC allocation across 20 projects in 20 states — targeting 87% of QEI in deep and severely distressed census tracts. Our pipeline focuses on healthcare and complementary high-impact sectors in markets where conventional capital is systematically absent.

[NARRATIVE PLACEHOLDER — Replace this text with your CDE's specific information. CDFI Fund reviewers score on specificity, evidence, and alignment with community need. Word limit for this section: 3000 words.]

 ...


### Section A — Markdown Rendering

In [6]:
from nmtcapp.sections.section_a_business import SectionABusinessStrategy

md_section = SectionABusinessStrategy().generate_markdown(app, analysis)
print(md_section[:800])

## Section A: Business Strategy

### Investment Thesis and Strategy

Heartland Impact CDE, LLC will deploy $65.0 million in NMTC allocation across 20 projects in 20 states — targeting 87% of QEI in deep and severely distressed census tracts. Our pipeline focuses on healthcare and complementary high-impact sectors in markets where conventional capital is systematically absent.

[NARRATIVE PLACEHOLDER — Replace this text with your CDE's specific information. CDFI Fund reviewers score on specificity, evidence, and alignment with community need. Word limit for this section: 3000 words.]



### Target Markets — Geographic and Demographic

Primary geographic targets: AZ, CA, FL, GA, IL and others.

Heartland Impact CDE, LLC's mission — "Deploy New Markets Tax Credit capital into deep-distress co


## 4. Table Builder Output

### Pipeline Table (Appendix A)

In [7]:
from nmtcapp.tables.pipeline_table import build_pipeline_table

df_pipeline = build_pipeline_table(pipeline, cde)
print(f"Shape: {df_pipeline.shape}")
cols = ["Project Name", "State", "Sector (NAICS)", "QEI Request ($)", "Distress Level", "NMTC Eligible (Y/N)"]
display(df_pipeline[cols].head(8))

Shape: (21, 33)


,Project Name,State,Sector (NAICS),QEI Request ($),Distress Level,NMTC Eligible (Y/N)
0,Southside Community Health Center,IL,621 – Ambulatory Health Care Services,8500000,Deep Distress,Y
1,East Houston Charter Academy,TX,611 – Educational Services,7000000,Deep Distress,Y
2,Bronx Food Hub,NY,722/336 – Food Services / Manufacturing,5000000,Severely Distressed,Y
3,Watts Manufacturing Center,CA,722/336 – Food Services / Manufacturing,4500000,Deep Distress,Y
4,Cleveland Neighborhood Clinic,OH,621 – Ambulatory Health Care Services,7500000,Deep Distress,Y
5,Atlanta Affordable Homes Phase II,GA,531 – Real Estate (Residential),9000000,Severely Distressed,Y
6,Miami Community Resource Center,FL,624 – Social Assistance / Community Facilities,4000000,Low Income Community,Y
7,North Philadelphia Wellness Hub,PA,621 – Ambulatory Health Care Services,8000000,Deep Distress,Y


### Distress Documentation (Appendix B)

In [8]:
from nmtcapp.tables.distress_table import build_distress_table

df_distress = build_distress_table(pipeline)
exclude = {"ACS Vintage", "CDFI Fund Source"}
display(df_distress[[c for c in df_distress.columns if c not in exclude]].head(6))

,Project ID,Project Name,"City, State",Census Tract (GEOID),NMTC Eligible,Distress Level,Severely Distressed Flag,NMTC Native Area,High Migration Rural (HMR),Opportunity Zone,Poverty Rate (%),Median Family Income,Unemployment Rate (%),Data Source
0,PRJ-001,Southside Community Health Center,"Chicago, IL",17031838200,Yes,Deep Distress,Yes,No,No,No,> 30%,See ACS,See ACS,CDFI Fund NMTC Eligibility Table (2016–2020 ACS)
1,PRJ-002,East Houston Charter Academy,"Houston, TX",48201223400,Yes,Deep Distress,Yes,No,No,Yes,> 30%,See ACS,See ACS,CDFI Fund NMTC Eligibility Table (2016–2020 ACS)
2,PRJ-003,Bronx Food Hub,"Bronx, NY",36005035900,Yes,Severely Distressed,Yes,No,No,Yes,> 20%,See ACS,See ACS,CDFI Fund NMTC Eligibility Table (2016–2020 ACS)
3,PRJ-004,Watts Manufacturing Center,"Los Angeles, CA",06037606200,Yes,Deep Distress,Yes,No,No,No,> 30%,See ACS,See ACS,CDFI Fund NMTC Eligibility Table (2016–2020 ACS)
4,PRJ-005,Cleveland Neighborhood Clinic,"Cleveland, OH",39035103200,Yes,Deep Distress,Yes,No,No,No,> 30%,See ACS,See ACS,CDFI Fund NMTC Eligibility Table (2016–2020 ACS)
5,PRJ-006,Atlanta Affordable Homes Phase II,"Atlanta, GA",13121008700,Yes,Severely Distressed,Yes,No,No,Yes,> 20%,See ACS,See ACS,CDFI Fund NMTC Eligibility Table (2016–2020 ACS)


### Geographic Targeting (Appendix C)

In [9]:
from nmtcapp.tables.geographic_table import build_geographic_table

df_geo = build_geographic_table(pipeline)
display(df_geo.head(10))

,State,Project Count,QEI ($),QEI (% of Total),Deep/Severe Projects,Native Area Projects,HMR Projects,OZ Projects
0,AZ,1,6500000.0,0.053061,1,1,0,0
1,CA,1,4500000.0,0.036735,1,0,0,0
2,FL,1,4000000.0,0.032653,0,0,0,0
3,GA,1,9000000.0,0.073469,1,0,0,1
4,IL,1,8500000.0,0.069388,1,0,0,0
5,IN,1,7500000.0,0.061224,1,0,0,0
6,KS,1,8500000.0,0.069388,0,0,0,0
7,LA,1,6000000.0,0.048980,1,0,0,1
8,MD,1,4200000.0,0.034286,1,0,0,0
9,MI,1,5000000.0,0.040816,1,0,0,1


### Impact Projections (Appendix D)

In [10]:
from nmtcapp.tables.impact_table import build_impact_table

df_impact = build_impact_table(pipeline)
cols = ["Project Name", "State", "Sector", "Jobs Created", "QEI ($)", "Cost per Job ($)"]
display(df_impact[cols].head(8))

,Project Name,State,Sector,Jobs Created,QEI ($),Cost per Job ($)
0,Southside Community Health Center,IL,Healthcare,52,8500000,240385
1,East Houston Charter Academy,TX,Education,38,7000000,257895
2,Bronx Food Hub,NY,Small Business,65,5000000,110769
3,Watts Manufacturing Center,CA,Small Business,80,4500000,75000
4,Cleveland Neighborhood Clinic,OH,Healthcare,42,7500000,238095
5,Atlanta Affordable Homes Phase II,GA,Affordable Housing,15,9000000,933333
6,Miami Community Resource Center,FL,Community Facility,28,4000000,196429
7,North Philadelphia Wellness Hub,PA,Healthcare,55,8000000,200000


## 5. Generate Full Application Package

In [11]:
output_dir = "../examples/sample_output"

paths = app.generate(
    output_dir=output_dir,
    formats=["markdown", "word", "excel", "pdf"],
)

print("Generated files:")
for fmt, path in paths.items():
    size_kb = os.path.getsize(path) // 1024
    print(f"  {fmt:10s}  {os.path.basename(path)}  ({size_kb} KB)")

Generated files:
  markdown    CDE-2018-0117_application.md  (31 KB)
  word        CDE-2018-0117_application.docx  (52 KB)
  excel       CDE-2018-0117_application.xlsx  (24 KB)
  pdf         CDE-2018-0117_application.pdf  (45 KB)


## 6. Verify Generated Files

### Markdown — first 1500 chars

In [12]:
with open(paths["markdown"]) as f:
    md_content = f.read()

print(md_content[:1500])

# NEW MARKETS TAX CREDIT ALLOCATION APPLICATION

**Heartland Impact CDE, LLC**

Application Round: CY2025  
Requested Allocation: **$65,000,000**  
Prepared: 2026-05-09  
Readiness Grade: **A** (86.6/100)  

---

*This document was generated by nmtc-application-builder v0.1. Narrative placeholders marked \[NARRATIVE PLACEHOLDER\] require CDE review and customization.*

---

## Executive Summary

Heartland Impact CDE, LLC respectfully requests $65.0MM in New Markets Tax Credit allocation for application round CY2025. Our 20-project pipeline spans 20 states with **87% of QEI committed to deep/severely distressed tracts** — ranking in the top quartile tier of historical NMTC applications.

**Application Readiness Score: 86.6/100 (Grade A)**

**Key Strengths:**
- High pipeline eligibility rate (≥80% score)
- Strong deep/severe distress concentration
- Good geographic diversity across multiple states

**Recommended Improvements Before Submission:**
- Add operating business projects (manufac

### Excel — sheet inventory

In [13]:
import openpyxl

wb = openpyxl.load_workbook(paths["excel"])
print("Excel sheets:")
for name in wb.sheetnames:
    ws = wb[name]
    print(f"  {name:30s}  ({ws.max_row} rows x {ws.max_column} cols)")

Excel sheets:
  Summary Dashboard               (26 rows x 6 cols)
  Pipeline Detail                 (24 rows x 33 cols)
  Distress Documentation          (24 rows x 15 cols)
  Geographic Targeting            (24 rows x 8 cols)
  Impact Projections              (24 rows x 18 cols)
  Investor Commitments            (6 rows x 8 cols)
  Track Record                    (7 rows x 6 cols)


### Word — paragraph and table count

In [14]:
from docx import Document

word_doc = Document(paths["word"])
print(f"Word document: {len(word_doc.paragraphs)} paragraphs, {len(word_doc.tables)} tables")

Word document: 141 paragraphs, 17 tables


### PDF — validity check

In [15]:
with open(paths["pdf"], "rb") as f:
    header = f.read(4)

size_kb = os.path.getsize(paths["pdf"]) // 1024
print(f"PDF header: {header}  (valid PDF: {header == b'%PDF'})")
print(f"PDF size:   {size_kb} KB")

PDF header: b'%PDF'  (valid PDF: True)
PDF size:   45 KB


## Summary

With a single `app.generate()` call, the library produces:

| Format | Contents |
|--------|----------|
| **Markdown** | Full application draft, version-control friendly |
| **Word** | Professional `.docx` with cover, sections A-E, 6 appendices |
| **Excel** | 7-sheet workbook with conditional formatting and charts |
| **PDF** | Board-ready PDF via ReportLab |

Each document includes:
- Cover page with CDE branding and readiness grade
- Executive summary with key metrics table
- Sections A-E with narrative, tables, and bullet lists
- Appendices: pipeline, distress, geographic, impact, track record, methodology

**Week 3** will add: win probability scoring, optimizer, HMDA integration, and Plotly visualizations.

See the [GitHub repo](https://github.com/Jaypatel1511/nmtc-application-builder) for the roadmap.